# Frengression versus Frugal Flows

This notebook runs the **same implementation** as [`compare_frugal_flows.py`](compare_frugal_flows.py). It compares the full interventional outcome distributions on one known binary-treatment, scalar-outcome simulation. The optional two-outcome study compares a joint Frengression fit to separate one-outcome Frengression fits. It is **not** a numerical comparison against a multivariate Frugal Flows implementation.

**No results are pre-filled.** Select a Python kernel for this repository in VS Code and run cells in order. The setup cell installs missing packages into that kernel's environment; initial installation and JAX compilation can take time. **Run All** performs a short smoke run, then the full benchmark. The full run can take hours, depending on hardware. Set `RUN_FULL = False` in the configuration cell if you only want the quick check.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# Works when VS Code uses either the repository root or examples/ as cwd.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "examples" / "compare_frugal_flows.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the frengression_pkg checkout in VS Code and use its notebook kernel.")

# Pinned to a Frugal Flows revision with sample_do and flexible_continuous.
FRUGAL_FLOWS_COMMIT = "26bc25cfa0c262e8a34af61841b202de79ffae0c"

def install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *map(str, args)])

if importlib.util.find_spec("frengression") is None:
    install("-e", ROOT)
if importlib.util.find_spec("frugal_flows") is None:
    install("git+https://github.com/llaurabatt/frugal-flows.git@" + FRUGAL_FLOWS_COMMIT)
missing = [name for name in ("scipy", "pandas", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    install(*missing)

sys.path.insert(0, str(ROOT / "examples"))
from compare_frugal_flows import run, simulate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print("Using repository:", ROOT)
print("Python kernel:", sys.executable)


## Check the simulation

Treatment assignment depends on three covariates. Both methods see identical observational training data. Separate draws from the known data-generating process provide the intervention truth. The two outcomes share covariates and noise, so their **joint** distribution contains information that separate scalar margins cannot represent.


In [ ]:
from scipy.stats import pearsonr

x_obs, z_obs, y_obs = simulate(1000, seed=2026, dim=2)
_, _, y0 = simulate(3000, seed=2027, intervention=0, dim=2)
_, _, y1 = simulate(3000, seed=2028, intervention=1, dim=2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(y0[:, 0], bins=45, density=True, alpha=0.55, label="do(0)")
axes[0].hist(y1[:, 0], bins=45, density=True, alpha=0.55, label="do(1)")
axes[0].set(xlabel="First outcome", ylabel="Density", title="Interventional margins")
axes[0].legend()
axes[1].scatter(y1[:500, 0], y1[:500, 1], s=7, alpha=0.35)
axes[1].set(xlabel="First outcome", ylabel="Second outcome",
            title=f"Joint do(1): correlation {pearsonr(y1[:, 0], y1[:, 1]).statistic:.2f}")
fig.tight_layout()
plt.show()


## Experiment settings

The small run below is only an integration check. `RUN_FULL = True` makes **Run All** continue to the five-seed benchmark after the smoke run; set it to `False` for a quick check only. `RUN_MULTIVARIATE` adds one joint two-outcome fit and two independent scalar fits per seed; it increases runtime. Results are saved in `examples/benchmark_outputs/`.


In [ ]:
from argparse import Namespace

RUN_FULL = True            # Set False to stop after the smoke run.
RUN_MULTIVARIATE = True  # Set False to run only the shared scalar comparison.
OUTPUT_DIR = ROOT / "examples" / "benchmark_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

smoke = Namespace(n=200, repeats=1, seed=2026, mc=100, truth_mc=1000,
                  fr_iters=20, flow_epochs=20, marginal_epochs=20,
                  methods=["frengression", "frugal_flows"], multivariate=False,
                  output=str(OUTPUT_DIR / "smoke.csv"))
full = Namespace(n=2000, repeats=5, seed=2026, mc=1000, truth_mc=50000,
                 fr_iters=2000, flow_epochs=2000, marginal_epochs=400,
                 methods=["frengression", "frugal_flows"],
                 multivariate=RUN_MULTIVARIATE,
                 output=str(OUTPUT_DIR / "comparison_results.csv"))
print("Full experiment enabled:", RUN_FULL)


## 1. Wiring check

This actually trains each scalar model briefly. Small training budgets mean these numbers **must not** be used in the paper. If installation, model API, or memory fails, send the full error message and Python/package versions for debugging.


In [ ]:
run(smoke)
smoke_results = pd.read_csv(smoke.output)
display(smoke_results)


## 2. Publication candidate run

With the default settings, **Run All** reaches this cell after the smoke check. A benchmark CSV is written after the fits finish; if a run fails partway through, the script currently does not checkpoint intermediate seeds. Inspect fit diagnostics and tune training budgets on a pilot *before* treating the five-seed summary as final.


In [ ]:
if RUN_FULL:
    run(full)
    results = pd.read_csv(full.output)
    display(results)
else:
    print("Skipped full run. Set RUN_FULL = True in the configuration cell, then rerun it and this cell.")


## 3. Compare across seeds

Lower is better for every error column. The scalar comparison is **head-to-head**. Fit time is diagnostic only: Frugal Flows times marginal CDF plus its flow, whereas Frengression times its `f,h` fit and omits `g`. The two-outcome comparison is **joint Frengression versus independently fitted Frengression margins**. It asks whether joint fitting recovers cross-outcome dependence; it does not compare to a multivariate Frugal Flows model.


In [ ]:
if RUN_FULL and Path(full.output).is_file():
    scalar = results.loc[results.setting == "binary_scalar"]
    scalar_metrics = ["ate_error", "wasserstein_mean", "energy_mean", "quantile_mae"]
    display(scalar.groupby("method")[scalar_metrics].agg(["mean", "std"]).round(4))
    if RUN_MULTIVARIATE:
        joint = results.loc[results.setting == "binary_two_outcomes"]
        joint_metrics = ["joint_energy_mean", "mean_error", "covariance_error"]
        display(joint.groupby("method")[joint_metrics].agg(["mean", "std"]).round(4))
else:
    print("Run the full experiment first; the one-seed smoke numbers are not paper results.")


## Interpretation and reproducibility

A useful strength claim would require consistently good **distributional** accuracy in the shared scalar task and, separately, smaller joint-distribution/covariance error for the joint fit while keeping marginal errors acceptable. Report all metrics, variation across seeds, the data-generating process, training budgets, hardware, and code versions, even if a competitor wins a metric. This is one simulation setting; add a distinct setting and assess sensitivity before claiming general superiority.

The procedure estimates interventional distributions under its stated simulation assumptions. Do not interpret the displayed seed percentiles as confidence intervals or use the two-outcome test as proof of correct longitudinal joint trajectories.


In [ ]:
import importlib.metadata as metadata

for package in ("frengression", "frugal-flows", "torch", "jax", "flowjax", "numpy", "scipy"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: version unavailable")
try:
    revision = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    print("frengression_pkg commit:", revision)
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print("Frugal Flows install target:", FRUGAL_FLOWS_COMMIT)
if RUN_FULL and Path(full.output).is_file():
    print("Share this CSV for interpretation:", full.output)
